# SQL Practice - Day 9

## Advanced CASE WHEN

10 Interview-Level Questions

## Q1

**Task:** Classify customers into Platinum, Gold, Silver and Bronze.

In [50]:
import pandas as pd
import pandasql
from pandasql import sqldf
df=pd.DataFrame({'customer_id':[101,102,103,104],'spending':[7000,4500,1800,700]})
df

,customer_id,spending
0,101,7000
1,102,4500
2,103,1800
3,104,700


In [51]:
sqldf("""
select customer_id ,spending,
case
    when spending >= 5000 then "Platinum"
    when spending >= 2000 then "GOLD"
    else "Bronze"
end as "Bucket"
from df



""")

,customer_id,spending,Bucket
0,101,7000,Platinum
1,102,4500,GOLD
2,103,1800,Bronze
3,104,700,Bronze


## Q2

**Task:** Conditional aggregation by region using CASE + SUM().

In [52]:
import pandas as pd
df=pd.DataFrame({'region':['North','South','East','West','North','East'],'sales':[100,200,300,400,500,600]})
df

,region,sales
0,North,100
1,South,200
2,East,300
3,West,400
4,North,500
5,East,600


In [53]:
sqldf("""
select 
sum(case when region = 'North' then sales else 0 end) as North,
sum(case when region = 'South' then sales else 0 end) as South,
sum(case when region = 'East' then sales else 0 end) as East,
sum(case when region = 'West' then sales else 0 end) as West
from df
""")

,North,South,East,West
0,600,200,900,400


## Q3

**Task:** Pivot Online and Offline revenue using CASE.

In [54]:
import pandas as pd
df=pd.DataFrame({'channel':['Online','Offline','Online','Offline'],'amount':[1000,800,1200,900]})
df

,channel,amount
0,Online,1000
1,Offline,800
2,Online,1200
3,Offline,900


In [55]:
sqldf("""
    select 
    sum(case when channel = "Online" then amount else 0 end ) as Online,
    sum(case when channel = "Offline" then amount else 0 end ) as Offline
    from df

""")

,Online,Offline
0,2200,1700


## Q4

**Task:** Loan risk classification using nested CASE.

In [56]:
import pandas as pd
df=pd.DataFrame({'loan_id':[1,2,3,4],'amount':[120000,150000,70000,90000],'overdue':['Y','N','Y','N']})
df

,loan_id,amount,overdue
0,1,120000,Y
1,2,150000,N
2,3,70000,Y
3,4,90000,N


In [57]:
sqldf("""
select
    loan_id,
    amount,
    overdue,
    case
        when amount >= 100000 and overdue = 'Y' then 'risk'
        when amount < 100000 and overdue = 'N' then 'no_risk'
        else 'moderate_risk'
    end as risk_status
from df
""")

,loan_id,amount,overdue,risk_status
0,1,120000,Y,risk
1,2,150000,N,moderate_risk
2,3,70000,Y,moderate_risk
3,4,90000,N,no_risk


## Q5

**Task:** Count Approved and Rejected loans.

In [58]:
import pandas as pd
df=pd.DataFrame({'status':['Approved','Rejected','Approved','Rejected','Approved']})
df

,status
0,Approved
1,Rejected
2,Approved
3,Rejected
4,Approved


In [59]:
sqldf("""
    select 
    count(case when status = 'Approved' then 1  end ) as count_approved,
    count(case when status = 'Rejected' then 1  end ) as count_rejected
    from df


""")

,count_approved,count_rejected
0,3,2


## Q6

**Task:** Label latest order using ROW_NUMBER() + CASE.

In [74]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,102],'order_date':['2025-01-01','2025-02-01','2025-01-15','2025-03-01']})
df

,customer_id,order_date
0,101,2025-01-01
1,101,2025-02-01
2,102,2025-01-15
3,102,2025-03-01


In [76]:
sqldf("""
with my_cte as (
    select customer_id , order_date,
    row_number()
    over(partition by customer_id order by order_date desc) as rn
    from df)
select customer_id, order_date,
    case
        when rn = 1 then "latest"
        else "old"
    end as Order_status
from my_cte
""")



,customer_id,order_date,Order_status
0,101,2025-02-01,latest
1,101,2025-01-01,old
2,102,2025-03-01,latest
3,102,2025-01-15,old


## Q7

**Task:** Bonus calculation using CASE.

In [79]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D'],'department':['Sales','IT','HR','Finance'],'salary':[60000,70000,50000,40000]})
df

,name,department,salary
0,A,Sales,60000
1,B,IT,70000
2,C,HR,50000
3,D,Finance,40000


In [81]:
sqldf("""
SELECT
    name,
    department,
    salary,
    CASE
        WHEN department = 'Sales' THEN salary * 0.20
        WHEN department = 'IT' THEN salary * 0.15
        WHEN department = 'HR' THEN salary * 0.10
        WHEN department = 'Finance' THEN salary * 0.05
        ELSE 0
    END AS bonus
FROM df

""")

,name,department,salary,bonus
0,A,Sales,60000,12000.0
1,B,IT,70000,10500.0
2,C,HR,50000,5000.0
3,D,Finance,40000,2000.0


## Q8

**Task:** High/Low performer based on department average.

In [82]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E'],'department':['HR','HR','IT','IT','IT'],'salary':[50000,70000,60000,90000,65000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,IT,60000
3,D,IT,90000
4,E,IT,65000


In [84]:
sqldf("""
SELECT
    name,
    department,
    salary,
    CASE
        WHEN salary >
             AVG(salary) OVER (PARTITION BY department)
        THEN 'High Performer'
        ELSE 'Low Performer'
    END AS performer_status
FROM df



""")

,name,department,salary,performer_status
0,A,HR,50000,Low Performer
1,B,HR,70000,High Performer
2,C,IT,60000,Low Performer
3,D,IT,90000,High Performer
4,E,IT,65000,Low Performer


## Q9

**Task:** Custom sorting using CASE in ORDER BY.

In [85]:
import pandas as pd
df=pd.DataFrame({'customer_id':[1,2,3,4],'tier':['Gold','Bronze','Premium','Silver']})
df

,customer_id,tier
0,1,Gold
1,2,Bronze
2,3,Premium
3,4,Silver


In [87]:
sqldf("""
SELECT
    customer_id,
    tier
FROM df
ORDER BY CASE
    WHEN tier = 'Premium' THEN 1
    WHEN tier = 'Gold' THEN 2
    WHEN tier = 'Silver' THEN 3
    WHEN tier = 'Bronze' THEN 4
    ELSE 5
END;

""")

,customer_id,tier
0,3,Premium
1,1,Gold
2,4,Silver
3,2,Bronze


## Q10

**Task:** KPI report using conditional aggregation.

In [88]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5],'amount':[500,1200,900,2000,1500]})
df

,order_id,amount
0,1,500
1,2,1200
2,3,900
3,4,2000
4,5,1500


In [91]:
sqldf("""
SELECT
    COUNT(*) AS total_orders,
    SUM(amount) AS total_revenue,
    SUM(CASE WHEN amount >= 1000 THEN 1 ELSE 0 END) AS high_value_orders,
    SUM(CASE WHEN amount < 1000 THEN 1 ELSE 0 END) AS low_value_orders,
    SUM(CASE WHEN amount >= 1000 THEN amount ELSE 0 END) AS high_value_revenue,
    SUM(CASE WHEN amount < 1000 THEN amount ELSE 0 END) AS low_value_revenue
FROM df;


""")

,total_orders,total_revenue,high_value_orders,low_value_orders,high_value_revenue,low_value_revenue
0,5,6100,3,2,4700,1400
